# Exercise P2.1: pandas Data Ingestion
### STAT 540 — Week 2


## Overview

In this exercise, you will read datasets in multiple formats using pandas, perform systematic inspection, and compare the Python workflow to the R workflow from Exercise R2.1.

## Task 1: Read a CSV File

In [ ]:
import pandas as pd

# From a URL
csv_data = pd.read_csv("https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv")

# Inspect
print(f"Shape: {csv_data.shape}")
print(f"\nColumn types:\n{csv_data.dtypes}")
csv_data.head()

Shape: (344, 7)

Column types:
species               object
island                object
bill_length_mm       float64
bill_depth_mm        float64
flipper_length_mm    float64
body_mass_g          float64
sex                   object
dtype: object


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,FEMALE


**Your turn:** What types did pandas assign to each column? Do any seem incorrect?

> Pandas assigned the Species, Island, and Sex columns to `object` types, and the measurable features to `float64`. I think the latter is okay, but `object` could probably be a more optimal data type, as there are a finite number of values for this variable.

## Task 2: Read an Excel File

In [ ]:
import pandas as pd

# Read default sheet
xlsx_data = pd.read_excel("/content/data/employees.xlsx")

# List all sheet names
xls = pd.ExcelFile("data/employees.xlsx")
print(f"Sheets: {xls.sheet_names}")

# Read all sheets into a dictionary
all_sheets = pd.read_excel("data/employees.xlsx", sheet_name=None)
for name, df in all_sheets.items():
    print(f"Sheet '{name}': {df.shape}")

xlsx_data.info()

Sheets: ['Sheet1']
Sheet 'Sheet1': (35, 9)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35 entries, 0 to 34
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   First Name              35 non-null     object 
 1   Last Name               35 non-null     object 
 2   Gender                  35 non-null     object 
 3   Age                     35 non-null     int64  
 4   Salary                  35 non-null     int64  
 5   Expenditure             35 non-null     int64  
 6   Savings                 35 non-null     int64  
 7   Expenditure Percentage  35 non-null     float64
 8   Savings Percentage      35 non-null     float64
dtypes: float64(2), int64(4), object(3)
memory usage: 2.6+ KB


## Task 3: Read and Inspect a JSON File

In [ ]:
import pandas as pd
import requests

# Read from an API (GitHub repos)
response = requests.get("https://api.github.com/users/hadley/repos?per_page=5")
repos = response.json()

# Flatten to DataFrame
df = pd.json_normalize(repos)

print(f"Shape: {df.shape}")
print(f"\nColumns ({len(df.columns)} total):")
print(df.columns.tolist())

# Preview a few key columns
df[["name", "full_name", "stargazers_count", "language"]].head()

Shape: (5, 104)

Columns (104 total):
['id', 'node_id', 'name', 'full_name', 'private', 'html_url', 'description', 'fork', 'url', 'forks_url', 'keys_url', 'collaborators_url', 'teams_url', 'hooks_url', 'issue_events_url', 'events_url', 'assignees_url', 'branches_url', 'tags_url', 'blobs_url', 'git_tags_url', 'git_refs_url', 'trees_url', 'statuses_url', 'languages_url', 'stargazers_url', 'contributors_url', 'subscribers_url', 'subscription_url', 'commits_url', 'git_commits_url', 'comments_url', 'issue_comment_url', 'contents_url', 'compare_url', 'merges_url', 'archive_url', 'downloads_url', 'issues_url', 'pulls_url', 'milestones_url', 'notifications_url', 'labels_url', 'releases_url', 'deployments_url', 'created_at', 'updated_at', 'pushed_at', 'git_url', 'ssh_url', 'clone_url', 'svn_url', 'homepage', 'size', 'stargazers_count', 'watchers_count', 'language', 'has_issues', 'has_projects', 'has_downloads', 'has_wiki', 'has_pages', 'has_discussions', 'forks_count', 'mirror_url', 'archived',

,name,full_name,stargazers_count,language
0,15-state-of-the-union,hadley/15-state-of-the-union,22,R
1,15-student-papers,hadley/15-student-papers,14,R
2,25-tidyverse-history,hadley/25-tidyverse-history,53,R
3,26-04-health-spending,hadley/26-04-health-spending,0,R
4,adv-r,hadley/adv-r,2458,TeX


**Your turn:** How many columns did `json_normalize` produce? Why are there so many compared to the original JSON fields?

> `pd.json_normalize()` produced 104 columns. There are significantly more columns than top-level JSON fields because the function flattens nested data structures. Whenever it encounters a nested dictionary (like the owner or license fields), it unpacks every nested key-value pair into its own distinct column, joining the parent and child keys with a dot (e.g., `owner.login`, `owner.id`, `license.name`).

## Task 4: Systematic Inspection

Pick one of your loaded datasets and run through the full inspection checklist:

In [ ]:
import pandas as pd

# Choose your dataset
df = csv_data   # or xlsx_data or json_data

# 1. Shape
print(f"Shape: {df.shape}")

# 2. Types and memory
df.info()

# 3. First and last rows
print("\nHead:")
print(df.head())
print("\nTail:")
print(df.tail())

# 4. Missing values
print("\nMissing values:")
print(df.isna().sum())
print(f"\nTotal missing: {df.isna().sum().sum()}")
print(f"Percent missing by column:\n{(df.isna().mean() * 100).round(1)}")

# 5. Summary statistics
print("\nNumeric summary:")
print(df.describe())

print("\nAll columns summary:")
print(df.describe(include="all"))

# 6. Unique values for categorical columns
for col in df.select_dtypes(include="object").columns:
    print(f"\n{col}: {df[col].nunique()} unique values")
    print(df[col].value_counts().head(5))

Shape: (344, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   species            344 non-null    object 
 1   island             344 non-null    object 
 2   bill_length_mm     342 non-null    float64
 3   bill_depth_mm      342 non-null    float64
 4   flipper_length_mm  342 non-null    float64
 5   body_mass_g        342 non-null    float64
 6   sex                333 non-null    object 
dtypes: float64(4), object(3)
memory usage: 18.9+ KB

Head:
  species     island  bill_length_mm  bill_depth_mm  flipper_length_mm  \
0  Adelie  Torgersen            39.1           18.7              181.0   
1  Adelie  Torgersen            39.5           17.4              186.0   
2  Adelie  Torgersen            40.3           18.0              195.0   
3  Adelie  Torgersen             NaN            NaN                NaN   
4  Adelie  Torgers

## Task 5: Compare R and Python Ingestion

Based on your experience in R2.1 and this exercise, fill in:

| Task | R Code | Python Code |
|------|--------|-------------|
| Read CSV | `read_csv("file.csv")` | `pd.read_csv("file.csv")` |
| Read Excel | `read_excel("file.xlsx")` | `pd.read_excel("file.xlsx")` |
| Read JSON from API | `fromJSON("url")` | `pd.json_normalize(requests.get("url").json())` |
| Check shape | `dim(df)` | `df.shape` |
| View types | `str(df) or glimpse(df)` | `df.info() or df.dtypes` |
| Count NAs | `colSums(is.na(df))` | `df.isna().sum()` |
| Summary stats | `summary(df)`| `df.describe()` |

**Your turn:** Which language felt more intuitive for data ingestion? Why?

> To be quite honest, the difference feels neglible, but I lean more towards R regarding intuitiveness, as the function names are easier to remember, and the library doesn't need to be added into the function call.

## Task 6: Write a Data Summary

For your chosen dataset, write a one-paragraph summary covering the same points as R2.1 Task 5: what the data represents, dimensions, variable types, missing values, and one thing you would investigate further.

> The employee dataset represents the demographic and financial profiles of a group of individuals, with dimensions of 35 observations (rows) across 9 variables (columns). The variables consist of a mix of data types: three categorical text variables (First Name, Last Name, Gender stored as object), four discrete numerical variables (Age, Salary, Expenditure, Savings stored as int64), and two continuous numerical variables (Expenditure Percentage, Savings Percentage stored as float64). According to the data info output, the dataset is perfectly complete with zero missing values across all fields. One interesting avenue for further investigation would be exploring how an employee's Salary or Age correlates with their Savings Percentage to see if specific demographic groups exhibit distinct financial behaviors.

## Submission

```bash
git add week02/exercises/P2.1*
git commit -m "Complete Exercise P2.1: pandas data ingestion across formats"
git push origin main
```